In [3]:
import pandas as pd
import geopandas as gpd

# 1. Load the LGA geographic file
# Note: If geopandas struggles with raw topojson, convert it to geojson first, 
# or use a standard LGA shapefile (.shp) from the ABS.
gdf = gpd.read_file('australia_lga.topojson')

# Calculate the center point (centroid) of each LGA polygon
gdf['lon'] = gdf.geometry.centroid.x
gdf['lat'] = gdf.geometry.centroid.y

# Keep only the name, lon, and lat columns (Adjust 'LGA_NAME21' to match your file's exact property name)
df_centroids = gdf[['LGA_NAME21', 'lon', 'lat']].copy()
df_centroids.rename(columns={'LGA_NAME21': 'LGA_Name'}, inplace=True)

# 2. Load the PHIDU Excel data (Sheet 3)
# Skipping the first 5 rows of metadata/titles so the actual data starts cleanly
file_path = 'phidu_data - AUST.xlsx'
df_hosp = pd.read_excel(file_path, sheet_name='Sheet3', skiprows=5, header=None)

# Extract only the columns we need based on the PHIDU structure
# Column 1 = LGA Name, Col 4 = All Circulatory, Col 9 = Ischaemic, Col 14 = Heart Failure, Col 19 = Stroke
df_clean = df_hosp[[1, 4, 9, 14, 19]].copy()
df_clean.columns = ['LGA_Name', 'All_Circulatory', 'Ischaemic', 'Heart_Failure', 'Stroke']

# Drop empty rows (like the state headers e.g., "NEW SOUTH WALES")
df_clean = df_clean.dropna(subset=['All_Circulatory'])

# 3. Clean LGA names to ensure they match perfectly before merging
# Sometimes PHIDU has " (C)" or " (A)" at the end of names, while TopoJSON doesn't. 
# This removes those tags if they exist.
df_clean['LGA_Name'] = df_clean['LGA_Name'].str.replace(r' \([A-Z]\)$', '', regex=True).str.strip()
df_centroids['LGA_Name'] = df_centroids['LGA_Name'].str.replace(r' \([A-Z]\)$', '', regex=True).str.strip()

# 4. Merge the geographical centroids with the health data
final_df = pd.merge(df_clean, df_centroids, on='LGA_Name', how='inner')

# 5. Export to the exact CSV format needed for the Vega-Lite map
final_df.to_csv('hospitalisations_lga_centroids.csv', index=False)
print("Data preparation complete! Saved to hospitalisations_lga_centroids.csv")

Data preparation complete! Saved to hospitalisations_lga_centroids.csv


In [5]:
import pandas as pd

# Load the dataset
df = pd.read_csv('hospitalisations_lga_centroids.csv')

# The hospitalisation count columns
cols_to_convert = ['All_Circulatory', 'Ischaemic', 'Heart_Failure', 'Stroke']

# Convert to numeric, round mathematically, and cast to integer
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).round().astype(int)

# Save the clean integer dataset
df.to_csv('hospitalisations_lga_centroids.csv', index=False)

In [6]:
import pandas as pd
import json

# 1. Load your TopoJSON file
with open('australia_lga.topojson', 'r', encoding='utf-8') as f:
    topo_data = json.load(f)

# 2. Extract the valid LGA names from the TopoJSON properties
# Note: The object name 'LGA_2021_AUST_GDA2020' might be slightly different 
# in your specific file (e.g., 'LGA_2021'). Check the "objects" key inside your TopoJSON if this throws an error.
topo_objects = topo_data['objects']
lga_layer_name = list(topo_objects.keys())[0] # Automatically gets the main layer name

valid_lgas = []
for geometry in topo_objects[lga_layer_name]['geometries']:
    # ABS 2021 TopoJSON usually uses 'LGA_NAME21'
    if 'properties' in geometry and 'LGA_NAME21' in geometry['properties']:
        valid_lgas.append(geometry['properties']['LGA_NAME21'])

# 3. Load your current hospitalisations CSV
df = pd.read_csv('hospitalisations_lga_centroids.csv')
original_count = len(df)

# 4. Filter the dataframe to KEEP ONLY the LGAs that exist in the TopoJSON
df_filtered = df[df['LGA_Name'].isin(valid_lgas)]
new_count = len(df_filtered)

# 5. Save the perfectly matched dataset
df_filtered.to_csv('hospitalisations_lga_centroids.csv', index=False)

print(f"Original LGAs in CSV: {original_count}")
print(f"LGAs kept (matched with map): {new_count}")
print(f"Removed {original_count - new_count} LGAs that were not in the TopoJSON.")
print("Saved as 'hospitalisations_lga_centroids.csv'")

Original LGAs in CSV: 540
LGAs kept (matched with map): 540
Removed 0 LGAs that were not in the TopoJSON.
Saved as 'hospitalisations_lga_centroids.csv'
